# Operator-learning comparison: kernel vs. neural

Combines the kernel-method results (`Framework{1,2}/results_*_no_pca.csv`) with the neural-operator
comparison CSV from `run_comparison.py` into a single table, so the kernelMO frameworks sit next to
the four neural settings (DeepONet no-coeff, DeepONet coeff-concat, MIONet, MNO).

The split matches across both families (first-N, no shuffle): Framework2 = 10000/4000,
Framework1 = 80%/20%. We also surface model size, training time, and inference time.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath('..')              # notebook lives in neural_network_experiments/
RESULTS_CSV = 'results_comparison.csv'         # produced by run_comparison.py

SETTING_ORDER = [
    'Kernel (vanilla)', 'Kernel (kernelMO)',
    'DeepONet (no coeff)', 'DeepONet (coeff concat)', 'MIONet', 'MNO',
]

def normalize_pde(name):
    """Join key so e.g. Framework2 'Param_wave' and kernel 'Param_Wave' align."""
    return str(name).lower()

## Load the neural-operator results

In [ ]:
neural = pd.read_csv(RESULTS_CSV)
if 'size' not in neural.columns:
    neural['size'] = 'medium'
neural['pde'] = neural['pde'].map(normalize_pde)
neural_cols = ['framework', 'pde', 'setting', 'size', 'split',
               'mean_relative_error', 'train_time_seconds', 'predict_time_seconds', 'n_params']
neural = neural[neural_cols]
neural.head()

## Load and tidy the kernel-method results

For each PDE / method / split we keep the best (lowest-error) kernel variant (RBF vs. Matern).
`method` maps to: `vanilla` -> *Kernel (vanilla)*, `product_gpr`/`framework1` -> *Kernel (kernelMO)*.

In [ ]:
KERNEL_GLOBS = {
    'Framework2': os.path.join(REPO_ROOT, 'Framework2', 'results_*_sample_no_pca.csv'),
    'Framework1': os.path.join(REPO_ROOT, 'Framework1', 'results_*_parameter_set_no_pca.csv'),
}
METHOD_LABEL = {
    'vanilla': 'Kernel (vanilla)',
    'product_gpr': 'Kernel (kernelMO)',
    'framework1': 'Kernel (kernelMO)',
}

def load_kernel_results():
    frames = []
    for framework, pattern in KERNEL_GLOBS.items():
        for path in glob.glob(pattern):
            df = pd.read_csv(path)
            df['framework'] = framework
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=neural_cols)
    k = pd.concat(frames, ignore_index=True)
    k = k[k['method'].isin(METHOD_LABEL)].copy()
    k['setting'] = k['method'].map(METHOD_LABEL)
    k['pde'] = k['pde'].map(normalize_pde)
    # best kernel variant per (framework, pde, setting, split)
    idx = k.groupby(['framework', 'pde', 'setting', 'split'])['mean_relative_error'].idxmin()
    k = k.loc[idx].copy()
    k['size'] = np.nan
    k['n_params'] = np.nan
    return k[neural_cols]

kernel = load_kernel_results()
kernel.head()

In [ ]:
combined = pd.concat([kernel, neural], ignore_index=True)
sizes_present = sorted(neural['size'].dropna().unique())
print('network sizes in neural results:', sizes_present)
combined.head()

## In-distribution test error (kernel vs. neural)

If multiple network sizes are present, the best (lowest-error) size is shown per neural setting; the
per-size breakdown is in the size-sweep section below.

In [ ]:
def error_pivot(split):
    sub = combined[combined['split'] == split]
    if sub.empty:
        return None
    p = sub.pivot_table(index=['framework', 'pde'], columns='setting',
                        values='mean_relative_error', aggfunc='min')
    cols = [c for c in SETTING_ORDER if c in p.columns]
    return p[cols]

test_pivot = error_pivot('test')
test_pivot.style.format('{:.3e}') if test_pivot is not None else 'no test rows'

## Out-of-distribution test error

In [ ]:
ood_pivot = error_pivot('ood')
ood_pivot.style.format('{:.3e}') if ood_pivot is not None else 'no ood rows'

## Bar charts per framework

Grouped bars: one cluster per PDE, one bar per method/setting, log-scaled relative error.

In [ ]:
def plot_split(split):
    sub = combined[combined['split'] == split]
    if sub.empty:
        print(f'no {split} rows')
        return
    frameworks = sorted(sub['framework'].unique())
    fig, axes = plt.subplots(1, len(frameworks), figsize=(8 * len(frameworks), 4.5), squeeze=False)
    for ax, fw in zip(axes[0], frameworks):
        p = sub[sub['framework'] == fw].pivot_table(index='pde', columns='setting',
                                                     values='mean_relative_error', aggfunc='min')
        cols = [c for c in SETTING_ORDER if c in p.columns]
        p = p[cols]
        pdes = list(p.index)
        x = np.arange(len(pdes))
        width = 0.8 / max(len(cols), 1)
        for i, c in enumerate(cols):
            ax.bar(x + i * width, p[c].values, width, label=c)
        ax.set_yscale('log')
        ax.set_xticks(x + width * (len(cols) - 1) / 2)
        ax.set_xticklabels(pdes, rotation=30, ha='right')
        ax.set_title(f'{fw} ({split})')
        ax.set_ylabel('mean relative L2 error')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_split('test')
plot_split('ood')

## Training and inference time

Both families record `train_time_seconds` (kernel = fit time; neural = full training run) and
`predict_time_seconds` (time to predict the test split). These are not apples-to-apples for training
(epochs vs. a single solve), but inference time is directly comparable. Neural rows use the medium
size unless only one size was run.

In [ ]:
tcols = ['train_time_seconds', 'predict_time_seconds', 'n_params']
tt = combined[combined['split'] == 'test'].copy()
# pick a single neural size for the headline timing table
ref_size = 'medium' if 'medium' in sizes_present else (sizes_present[0] if sizes_present else None)
tt = tt[tt['size'].isna() | (tt['size'] == ref_size)]
timing = tt.pivot_table(index=['framework', 'pde'], columns='setting', values='predict_time_seconds', aggfunc='min')
timing = timing[[c for c in SETTING_ORDER if c in timing.columns]]
print('Inference time on the test split (seconds), neural size =', ref_size)
timing.style.format('{:.3f}')

In [ ]:
train_t = tt.pivot_table(index=['framework', 'pde'], columns='setting', values='train_time_seconds', aggfunc='min')
train_t = train_t[[c for c in SETTING_ORDER if c in train_t.columns]]
print('Training time (seconds), neural size =', ref_size)
train_t.style.format('{:.2f}')

## Network-size sweep (neural only)

Only meaningful when `run_comparison.py --sizes small medium large` was used. Shows how error,
parameter count, and training/inference time scale with network size, averaged over PDEs.

In [ ]:
SIZE_ORDER = ['small', 'medium', 'large']
n_test = neural[neural['split'] == 'test'].copy()
if n_test['size'].nunique() <= 1:
    print('Only one network size present — rerun with `--sizes small medium large` to populate this section.')
else:
    order = [s for s in SIZE_ORDER if s in n_test['size'].unique()]
    summary = (n_test.groupby(['setting', 'size'])
               .agg(mean_rel_err=('mean_relative_error', 'mean'),
                    n_params=('n_params', 'mean'),
                    train_s=('train_time_seconds', 'mean'),
                    predict_s=('predict_time_seconds', 'mean'))
               .reset_index())
    display(summary.pivot(index='setting', columns='size', values='mean_rel_err')[order].style.format('{:.3e}'))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for setting, g in summary.groupby('setting'):
        g = g.set_index('size').reindex(order).reset_index()
        axes[0].plot(g['n_params'], g['mean_rel_err'], 'o-', label=setting)
        axes[1].plot(g['n_params'], g['predict_s'], 'o-', label=setting)
    for ax, ylab in zip(axes, ['mean relative L2 error (test)', 'inference time (s, test)']):
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('trainable parameters'); ax.set_ylabel(ylab)
        ax.legend(fontsize=8)
    axes[0].set_title('Accuracy vs. size'); axes[1].set_title('Inference cost vs. size')
    plt.tight_layout(); plt.show()